In [2]:
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
import torch
from torch import nn


df = pd.read_csv("iris.data", header=None)
df = df[df[4] != "Iris-setosa"]

X = df.iloc[:, :4].to_numpy()
y = (df[4] == "Iris-virginica").astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:


model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=0)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Keras test doğruluğu: {accuracy:.2%}")

Keras test doğruluğu: 90.00%


In [6]:


Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_test, dtype=torch.float32)

net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

for epoch in range(20):
    optimizer.zero_grad()
    loss = criterion(net(Xtr), ytr)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    predictions = (torch.sigmoid(net(Xte)) >= 0.5).int().numpy().ravel()

print(f"PyTorch test doğruluğu: {(predictions == y_test).mean():.2%}")

PyTorch test doğruluğu: 75.00%
